In [1]:
%reload_ext autoreload
%autoreload 2
%aimport rdflib_reasoning

# Demo: RDFS Inference and Proof Reconstruction

This notebook demonstrates the current end-to-end reasoning path at the RDFLib level:

- add asserted triples to an RDFLib `Dataset`
- let `RETEStore` materialize inferred triples through `RETEEngine`
- capture engine-native `DerivationRecord` values during inference
- reconstruct a `DirectProof` for one inferred triple

The explanation step is still somewhat manual: we inject a recording derivation logger into `RETEEngineFactory`, then pass those records to `DerivationProofReconstructor`.

In [2]:
from pprint import pprint

from rdflib import Dataset, URIRef
from rdflib.namespace import RDF, RDFS
from rdflib.plugins.stores.memory import Memory
from rdflib_reasoning.engine import (
    RDFS_RULES,
    DerivationLogger,
    DerivationProofReconstructor,
    DerivationRecord,
    RETEEngineFactory,
    TripleFact,
)
from rdflib_reasoning.engine.rete_store import RETEStore

In [3]:
from rdflib import Graph, Node

graph: Graph


def pretty(triple: tuple[Node, Node, Node]) -> tuple[str, str, str]:
    # NOTE: This is **NOT** the canonical way to pretty-print derivations / proofs
    # TODO: Replace this after `Proof rendering` feature is available.
    return tuple(graph.qname(n) for n in triple)


class RecordingLogger(DerivationLogger):
    def __init__(self) -> None:
        self.records: list[DerivationRecord] = []

    def record(self, record: DerivationRecord) -> None:
        if (
            len(record.conclusions) == 1
            and isinstance((fact := record.conclusions[0]), TripleFact)
            and not record.silent
        ):
            print(f"Recording {fact.kind} derivation: {pretty(fact.triple)}")
        self.records.append(record)


logger = RecordingLogger()
factory = RETEEngineFactory(rules=RDFS_RULES, derivation_logger=logger)
store = RETEStore(Memory(), factory)
dataset = Dataset(store=store)
graph = dataset.default_graph
graph.bind("ex", "urn:test:")

In [ ]:
alice = URIRef("urn:test:alice")
person = URIRef("urn:test:Person")
mammal = URIRef("urn:test:Mammal")
animal = URIRef("urn:test:Animal")

graph.add((alice, RDF.type, person))
print("-" * 80)
graph.add((person, RDFS.subClassOf, mammal))
print("-" * 80)
graph.add((mammal, RDFS.subClassOf, animal))
print("-" * 80)

sorted(pretty(t) for t in graph.triples((alice, RDF.type, None)))

--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
Recording triple derivation: ('ex:alice', 'rdf:type', 'ex:Mammal')
--------------------------------------------------------------------------------
Recording triple derivation: ('ex:alice', 'rdf:type', 'ex:Animal')
Recording triple derivation: ('ex:Person', 'rdfs:subClassOf', 'ex:Animal')
--------------------------------------------------------------------------------


[('ex:alice', 'rdf:type', 'ex:Animal'),
 ('ex:alice', 'rdf:type', 'ex:Mammal'),
 ('ex:alice', 'rdf:type', 'ex:Person')]

In [5]:
goal = TripleFact(context=graph.identifier, triple=(alice, RDF.type, animal))
print("Goal triple materialized:", goal.triple in graph)

Goal triple materialized: True


In [6]:
proof = DerivationProofReconstructor().reconstruct(goal, logger.records)

print("Verdict:", proof.verdict)
print("Top-level node kind:", proof.proof.node_kind)

Verdict: proved
Top-level node kind: rule_application


### Displaying Proofs - Mermaid Diagram

In [7]:
from rdflib_reasoning.engine.proof_notebook import display_proof_mermaid

display_proof_mermaid(proof, namespace_manager=graph.namespace_manager)

```mermaid
flowchart TD
n2[["rdfs:rdfs9
IF (?x, rdfs:subClassOf, ?y) AND (?a, rdf:type, ?x) AND different_terms(?x, ?y)
THEN (?a, rdf:type, ?y)"]]
n3("(ex:alice rdf:type ex:Animal)")
n3 -->|entailed_by| n2
n4["(ex:Mammal rdfs:subClassOf ex:Animal)"]
n2 -->|had_premise| n4
n5[["rdfs:rdfs9
IF (?x, rdfs:subClassOf, ?y) AND (?a, rdf:type, ?x) AND different_terms(?x, ?y)
THEN (?a, rdf:type, ?y)"]]
n6("(ex:alice rdf:type ex:Mammal)")
n6 -->|entailed_by| n5
n7["(ex:Person rdfs:subClassOf ex:Mammal)"]
n5 -->|had_premise| n7
n8["(ex:alice rdf:type ex:Person)"]
n5 -->|had_premise| n8
n2 -->|had_premise| n6
n1>"(ex:alice rdf:type ex:Animal)"]
n1 -->|justified_by| n3
```

### Displaying Proofs - Markdown Text

In [ ]:
from rdflib_reasoning.engine.proof_notebook import display_proof_markdown

display_proof_markdown(proof, namespace_manager=graph.namespace_manager)

## Direct Proof

- Context: `<urn:x-rdflib:default>`
- Goal: `(ex:alice rdf:type ex:Animal)`
- Verdict: `proved`

### Steps

- Rule Application: `rdfs:rdfs9`
  - Conclusions:
    - `(ex:alice rdf:type ex:Animal)`
  - Premises:
    - Leaf: `(ex:Mammal rdfs:subClassOf ex:Animal)`
    - Rule Application: `rdfs:rdfs9`
      - Conclusions:
        - `(ex:alice rdf:type ex:Mammal)`
      - Premises:
        - Leaf: `(ex:Person rdfs:subClassOf ex:Mammal)`
        - Leaf: `(ex:alice rdf:type ex:Person)`

## Proofs Are Pydantic Models

In [9]:
pprint(proof.model_dump(mode="python"))

{'context': rdflib.term.URIRef('urn:x-rdflib:default'),
 'goal': {'context': rdflib.term.URIRef('urn:x-rdflib:default'),
          'kind': 'triple',
          'triple': (rdflib.term.URIRef('urn:test:alice'),
                     rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
                     rdflib.term.URIRef('urn:test:Animal'))},
 'notes': None,
 'proof': {'conclusions': [{'context': rdflib.term.URIRef('urn:x-rdflib:default'),
                            'kind': 'triple',
                            'triple': (rdflib.term.URIRef('urn:test:alice'),
                                       rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
                                       rdflib.term.URIRef('urn:test:Animal'))}],
           'derivation': {'bindings': [{'name': 'a',
                                        'value': rdflib.term.URIRef('urn:test:alice')},
                                       {'name': 'x',
                               

In [10]:
print("Recorded derivations:", len(logger.records))
for record in logger.records:
    print(record.rule_id, [fact.triple for fact in record.conclusions])

Recorded derivations: 18
ruleset='rdfs' rule_id='rdfs1' [(rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#Property'))]
ruleset='rdfs' rule_id='rdfs4a' [(rdflib.term.URIRef('urn:test:alice'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#Resource'))]
ruleset='rdfs' rule_id='rdfs4b' [(rdflib.term.URIRef('urn:test:Person'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#Resource'))]
ruleset='rdfs' rule_id='rdfs4a' [(rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#Resource'))]
ruleset='rdfs' rule_id='rdfs4b' [(rdflib.term.URIRe

## Notes

- The graph materialization is driven entirely through the RDFLib store integration path.
- The proof reconstruction currently uses derivation logs, not a public `graph.explain(...)` API.
- If multiple derivation records can justify the same goal, the current reconstructor chooses the shallowest matching derivation.